# Use this notebook for extracting each clip from the YASHSET vibration recordings.
For now, only data from port_3331 since that sensor was the closest to the audio source.
Extracting clips from other sensors will probably need more tinkering of the parameters in the last cell.

## Locate the dataset directory

In [1]:
import numpy as np
import os
import glob

# 1. Setup paths
folder_path = 'C:/Users/ShuBe/triple_05_03/port_3331'
sample_rate = 7400

# 2. Get all .npy files and sort them
# Sorting ensures the "time" suffix in your filenames keeps the audio in order
files = sorted(glob.glob(os.path.join(folder_path, "vibration_*.npy")))

if not files:
    print("No files found! Check your directory path.")
else:
    print(f"Found {len(files)} files. Processing...")

    # 3. Load and concatenate
    audio_segments = []
    for f in files:
        segment = np.load(f)
        audio_segments.append(segment)
    
    # Combine all 1-second arrays into one long array
    full_audio = np.concatenate(audio_segments)

Found 1812 files. Processing...


#### OPTIONAL Export data as a .wav file for listening
Useful for making sure the data sounds okay for the most part

In [6]:
from scipy.io import wavfile

output_filename = 'test.wav'
# 4. Normalize (Prevents ear-piercing distortion/clipping)
wav_audio = full_audio - np.mean(full_audio)
# Scales the data to fit between -1.0 and 1.0
max_amplitude = np.max(np.abs(wav_audio))
if max_amplitude > 0:
    wav_audio = wav_audio / max_amplitude

# 5. Export to WAV
wavfile.write(output_filename, sample_rate, wav_audio.astype(np.float32))
print(f"Success! Saved {len(files)} seconds of audio to {output_filename}")

Success! Saved 1812 seconds of audio to test.wav


#### Iterates through the full_audio object created by the first cell above and detects where in the data there is a strong 300 Hz signal.
Change dominance_ratio_min and volume_z_min if trying to extract the other ports

In [2]:
def detect_300hz_dominant_frames(
    audio,
    sample_rate=7400,
    target_hz=300.0,
    frame_s=0.20,
    hop_s=0.05,
    freq_tolerance_hz=20.0,
    dominance_ratio_min=1.8,
    volume_z_min=1.0,
):
    """
    Detect frames where:
      1) 300 Hz is the dominant spectral component
      2) raw signal volume is above baseline

    Returns:
      frame_centers_s: center time of each frame
      is_candidate: bool mask for candidate beep frames
      dominant_freqs_hz: dominant frequency per frame
      dominance_ratio: target-band power / median non-target-bin power
      frame_rms: raw-data RMS per frame
    """
    x = np.asarray(audio, dtype=np.float64)
    if x.ndim != 1:
        raise ValueError("audio must be 1D")

    # remove DC only (keep raw amplitude behavior)
    x = x - np.mean(x)

    frame_n = int(round(frame_s * sample_rate))
    hop_n = max(1, int(round(hop_s * sample_rate)))
    if frame_n < 16 or len(x) < frame_n:
        return np.array([]), np.array([], dtype=bool), np.array([]), np.array([]), np.array([])

    window = np.hanning(frame_n)
    freqs = np.fft.rfftfreq(frame_n, d=1.0 / sample_rate)

    target_mask = (freqs >= target_hz - freq_tolerance_hz) & (freqs <= target_hz + freq_tolerance_hz)
    valid_mask = freqs >= 40.0  # ignore very-low bins for dominance voting
    non_target_mask = valid_mask & (~target_mask)

    centers_s = []
    dominant_freqs = []
    dom_ratio = []
    frame_rms = []

    for i in range(0, len(x) - frame_n + 1, hop_n):
        frame_raw = x[i : i + frame_n]
        frame_win = frame_raw * window

        spec = np.fft.rfft(frame_win)
        mag = np.abs(spec)

        # dominant frequency in useful range
        mag_valid = mag.copy()
        mag_valid[~valid_mask] = 0.0
        dom_idx = int(np.argmax(mag_valid))
        dom_f = freqs[dom_idx]

        # how much stronger target band is than background bins
        target_power = float(np.mean(mag[target_mask] ** 2)) if np.any(target_mask) else 0.0
        bg = mag[non_target_mask] ** 2
        bg_level = float(np.median(bg)) + 1e-12 if bg.size > 0 else 1e-12
        ratio = target_power / bg_level

        rms = float(np.sqrt(np.mean(frame_raw**2)))

        centers_s.append((i + frame_n // 2) / sample_rate)
        dominant_freqs.append(dom_f)
        dom_ratio.append(ratio)
        frame_rms.append(rms)

    centers_s = np.asarray(centers_s)
    dominant_freqs = np.asarray(dominant_freqs)
    dom_ratio = np.asarray(dom_ratio)
    frame_rms = np.asarray(frame_rms)

    # robust volume threshold from raw RMS
    rms_med = float(np.median(frame_rms))
    rms_mad = float(np.median(np.abs(frame_rms - rms_med)) + 1e-12)
    volume_thr = rms_med + volume_z_min * rms_mad

    freq_ok = np.abs(dominant_freqs - target_hz) <= freq_tolerance_hz
    ratio_ok = dom_ratio >= dominance_ratio_min
    volume_ok = frame_rms >= volume_thr

    is_candidate = freq_ok & ratio_ok & volume_ok
    return centers_s, is_candidate, dominant_freqs, dom_ratio, frame_rms


def candidate_frames_to_regions(frame_centers_s, candidate_mask, max_gap_s=0.20):
    """Merge neighboring candidate frames into time regions."""
    if len(frame_centers_s) == 0:
        return []

    idx = np.where(candidate_mask)[0]
    if idx.size == 0:
        return []

    regions = []
    start_i = idx[0]
    prev_i = idx[0]

    for k in idx[1:]:
        if frame_centers_s[k] - frame_centers_s[prev_i] <= max_gap_s:
            prev_i = k
            continue

        regions.append((float(frame_centers_s[start_i]), float(frame_centers_s[prev_i])))
        start_i = k
        prev_i = k

    regions.append((float(frame_centers_s[start_i]), float(frame_centers_s[prev_i])))
    return regions

In [3]:
# Run unconstrained 300 Hz + volume detection (no period locking)
centers_s, is_candidate, dominant_freqs_hz, dominance_ratio, frame_rms = detect_300hz_dominant_frames(
    full_audio,
    sample_rate=sample_rate,
    target_hz=300.0,
    frame_s=0.20,
    hop_s=0.05,
    freq_tolerance_hz=10.0,
    dominance_ratio_min=10000.0,
    volume_z_min=4.0,
)

candidate_regions = candidate_frames_to_regions(centers_s, is_candidate, max_gap_s=0.20)

print(f"Total frames: {len(centers_s)}")
print(f"Candidate frames: {int(np.sum(is_candidate))}")
print(f"Candidate regions: {len(candidate_regions)}")

if len(candidate_regions) > 0:
    durations = np.array([b - a for a, b in candidate_regions])
    print(f"Candidate region duration min/max: {durations.min():.3f}s / {durations.max():.3f}s")

def _fmt_mmss(t_sec):
    m = int(t_sec // 60)
    s = t_sec - 60 * m
    return f"{m:02d}:{s:06.3f}"


print("\nFirst 50 candidate regions (start -> end, mm:ss):")
for i, (a, b) in enumerate(candidate_regions[:50], 1):
    print(f"{i:02d}: {_fmt_mmss(a)} -> {_fmt_mmss(b)}")

Total frames: 34698
Candidate frames: 751
Candidate regions: 38
Candidate region duration min/max: 0.100s / 1.050s

First 50 candidate regions (start -> end, mm:ss):
01: 00:03.300 -> 00:04.350
02: 00:45.450 -> 00:46.200
03: 01:32.650 -> 01:33.400
04: 02:20.500 -> 02:21.500
05: 03:06.500 -> 03:07.500
06: 03:51.700 -> 03:52.750
07: 04:35.250 -> 04:36.300
08: 05:23.450 -> 05:24.450
09: 06:11.900 -> 06:12.900
10: 07:00.500 -> 07:01.500
11: 07:49.000 -> 07:50.050
12: 08:37.550 -> 08:38.600
13: 09:25.900 -> 09:26.900
14: 10:14.400 -> 10:15.450
15: 11:02.850 -> 11:03.850
16: 11:50.950 -> 11:51.950
17: 12:37.900 -> 12:38.900
18: 13:20.900 -> 13:21.950
19: 14:07.950 -> 14:08.950
20: 14:54.400 -> 14:55.450
21: 15:41.300 -> 15:42.300
22: 16:28.800 -> 16:29.600
23: 17:17.300 -> 17:18.350
24: 18:05.850 -> 18:06.850
25: 18:54.200 -> 18:55.200
26: 19:41.500 -> 19:42.500
27: 20:29.500 -> 20:30.500
28: 21:14.950 -> 21:15.950
29: 21:59.350 -> 22:00.300
30: 22:43.800 -> 22:44.800
31: 23:29.450 -> 23:30.4

In [4]:
# Build vibration clips between beeps using candidate region starts.
# Rule:
#   clip_start = current_beep_start + 1.5 s
#   clip_end   = next_beep_start - 0.5 s

beep_starts_s = np.array([start for start, _ in candidate_regions], dtype=float)

clip_start_offset_s = 1.5
clip_end_before_next_s = 0.25

between_beep_clips = []          # list of 1D numpy arrays
between_beep_ranges_s = []       # list of (clip_start_s, clip_end_s)
between_beep_ranges_idx = []     # list of (start_idx, end_idx)

for i in range(len(beep_starts_s) - 1):
    clip_start_s = beep_starts_s[i] + clip_start_offset_s
    clip_end_s = beep_starts_s[i + 1] - clip_end_before_next_s

    # Skip invalid/negative-length windows.
    if clip_end_s <= clip_start_s:
        continue

    start_idx = int(round(clip_start_s * sample_rate))
    end_idx = int(round(clip_end_s * sample_rate))

    start_idx = max(0, min(start_idx, len(full_audio)))
    end_idx = max(0, min(end_idx, len(full_audio)))

    if end_idx <= start_idx:
        continue

    between_beep_clips.append(full_audio[start_idx:end_idx])
    between_beep_ranges_s.append((clip_start_s, clip_end_s))
    between_beep_ranges_idx.append((start_idx, end_idx))


def _fmt_mmss(t_sec):
    m = int(t_sec // 60)
    s = t_sec - 60 * m
    return f"{m:02d}:{s:06.3f}"

print(f"Extracted between-beep clips: {len(between_beep_clips)}")
if between_beep_ranges_s:
    lengths_s = np.array([b - a for a, b in between_beep_ranges_s])
    print(f"Clip duration min/max: {lengths_s.min():.3f}s / {lengths_s.max():.3f}s")

print("\nFirst 20 clip ranges (start -> end, mm:ss):")
for i, (a, b) in enumerate(between_beep_ranges_s[:20], 1):
    print(f"{i:02d}: {_fmt_mmss(a)} -> {_fmt_mmss(b)}")

Extracted between-beep clips: 37
Clip duration min/max: 40.400s / 46.850s

First 20 clip ranges (start -> end, mm:ss):
01: 00:04.800 -> 00:45.200
02: 00:46.950 -> 01:32.400
03: 01:34.150 -> 02:20.250
04: 02:22.000 -> 03:06.250
05: 03:08.000 -> 03:51.450
06: 03:53.200 -> 04:35.000
07: 04:36.750 -> 05:23.200
08: 05:24.950 -> 06:11.650
09: 06:13.400 -> 07:00.250
10: 07:02.000 -> 07:48.750
11: 07:50.500 -> 08:37.300
12: 08:39.050 -> 09:25.650
13: 09:27.400 -> 10:14.150
14: 10:15.900 -> 11:02.600
15: 11:04.350 -> 11:50.700
16: 11:52.450 -> 12:37.650
17: 12:39.400 -> 13:20.650
18: 13:22.400 -> 14:07.700
19: 14:09.450 -> 14:54.150
20: 14:55.900 -> 15:41.050


In [5]:
import os
import csv
import numpy as np
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
for _p in [_cwd, *_cwd.parents]:
    if (_p / "repo_paths.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError("Could not find repo_paths.py; open the project folder in Jupyter.")

from repo_paths import find_repo_root

REPO_ROOT = find_repo_root()

# Export between-beep clips as .npy and assign labels by order from timestamps.csv
output_dir = str(REPO_ROOT / "dataset")
index_csv_path = os.path.join(output_dir, "index.csv")
os.makedirs(output_dir, exist_ok=True)

# Class mapping
label_to_id = {
    "background": 0,
    "calm_speech": 1,
    "distress": 2,
}

# Read labels from timestamps.csv in row order
rows = []
with open(REPO_ROOT / "timestamps.csv", "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for r in reader:
        class_name = (r.get("class_name") or "").strip()
        if class_name in label_to_id:
            rows.append(r)

num_clips = len(between_beep_clips)
num_labels = len(rows)
num_used = min(num_clips, num_labels)

if num_used == 0:
    raise RuntimeError("No clips or no usable labels found.")

if num_clips != num_labels:
    print(
        f"Warning: clip count ({num_clips}) != label rows ({num_labels}). "
        f"Using first {num_used} by order."
    )

# Create class subdirectories
for class_name in label_to_id.keys():
    os.makedirs(os.path.join(output_dir, class_name), exist_ok=True)

# Write .npy files + index.csv
with open(index_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "clip_idx",
        "file_path",
        "label_id",
        "label_name",
        "start_s",
        "end_s",
        "num_samples",
    ])

    for i in range(num_used):
        clip = np.asarray(between_beep_clips[i], dtype=np.float32)
        start_s, end_s = between_beep_ranges_s[i]

        label_name = rows[i]["class_name"].strip()
        label_id = label_to_id[label_name]

        filename = f"clip_{i:04d}.npy"
        file_path = os.path.join(output_dir, label_name, filename)
        np.save(file_path, clip)

        writer.writerow([
            i,
            file_path.replace("\\", "/"),
            label_id,
            label_name,
            f"{start_s:.6f}",
            f"{end_s:.6f}",
            len(clip),
        ])

print(f"Saved {num_used} labeled clips to: {output_dir}")
print(f"Index CSV: {index_csv_path}")

# Quick label distribution check
counts = {k: 0 for k in label_to_id.keys()}
for i in range(num_used):
    counts[rows[i]["class_name"].strip()] += 1
print("Label counts:", counts)

Saved 37 labeled clips to: dataset
Index CSV: dataset\index.csv
Label counts: {'background': 10, 'calm_speech': 13, 'distress': 14}


In [36]:
import csv
import numpy as np
import torch
from torch.utils.data import Dataset


class VibrationDataset(Dataset):
    """
    PyTorch dataset for saved .npy vibration clips.

    Features:
      - Optional fixed-length crop (e.g., 6 seconds)
      - Random crop for training, center crop for eval
      - Zero-padding if clip is shorter than requested crop
      - Optional per-sample z-score normalization
    """

    def __init__(
        self,
        index_csv,
        sample_rate=7400,
        crop_seconds=None,
        random_crop=False,
        normalize="zscore",  # "zscore" or None
        transform=None,
        dtype=np.float32,
    ):
        self.index_csv = index_csv
        self.sample_rate = sample_rate
        self.crop_seconds = crop_seconds
        self.random_crop = random_crop
        self.normalize = normalize
        self.transform = transform
        self.dtype = dtype
        self.records = []

        with open(index_csv, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.records.append(row)

        if len(self.records) == 0:
            raise RuntimeError(f"No records found in {index_csv}")

        self.crop_n = None
        if crop_seconds is not None:
            self.crop_n = int(round(crop_seconds * sample_rate))
            if self.crop_n <= 0:
                raise ValueError("crop_seconds must be positive")

    def __len__(self):
        return len(self.records)

    def _crop_or_pad(self, x):
        if self.crop_n is None:
            return x

        n = len(x)
        if n > self.crop_n:
            max_start = n - self.crop_n
            if self.random_crop:
                start = np.random.randint(0, max_start + 1)
            else:
                start = max_start // 2
            x = x[start : start + self.crop_n]
        elif n < self.crop_n:
            pad_n = self.crop_n - n
            x = np.pad(x, (0, pad_n), mode="constant")

        return x

    def _normalize(self, x):
        if self.normalize == "zscore":
            mu = float(np.mean(x))
            sigma = float(np.std(x))
            x = (x - mu) / (sigma + 1e-8)
        return x

    def __getitem__(self, idx):
        row = self.records[idx]

        x = np.load(row["file_path"]).astype(self.dtype)
        y = int(row["label_id"])

        x = self._crop_or_pad(x)
        x = self._normalize(x)

        # Shape: [1, T] for 1D conv models
        x = torch.from_numpy(x).unsqueeze(0)
        y = torch.tensor(y, dtype=torch.long)

        if self.transform is not None:
            x = self.transform(x)

        return x, y


# Example datasets: random crop for train, deterministic center crop for eval
train_ds = VibrationDataset(
    index_csv_path,
    sample_rate=sample_rate,
    crop_seconds=5.0,
    random_crop=True,
    normalize="zscore",
)

val_ds = VibrationDataset(
    index_csv_path,
    sample_rate=sample_rate,
    crop_seconds=5.0,
    random_crop=False,
    normalize="zscore",
)

print("Train dataset size:", len(train_ds))
print("Val dataset size:", len(val_ds))

# Peek at one sample
x0, y0 = train_ds[0]
print("Train sample shape:", tuple(x0.shape), "label_id:", int(y0))

Train dataset size: 37
Val dataset size: 37
Train sample shape: (1, 37000) label_id: 1


In [7]:
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
for _p in [_cwd, *_cwd.parents]:
    if (_p / "repo_paths.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError("Could not find repo_paths.py; open the project folder in Jupyter.")

from repo_paths import ensure_utils_importable

ensure_utils_importable()

from vibration_data import make_train_val_datasets

# One split config -> two dataset objects with split-specific behavior
train_ds, val_ds = make_train_val_datasets(
    index_csv_path,
    train_ratio=0.8,
    seed=42,
    sample_rate=sample_rate,
    window_seconds=5.0,
    normalize="zscore",
)

print("Train clips:", len(train_ds))
print("Val 5s segments:", len(val_ds))

train_counts = {}
for r in train_ds.records:
    train_counts[r["label_name"]] = train_counts.get(r["label_name"], 0) + 1

val_clip_counts = {}
for r in val_ds.records:
    val_clip_counts[r["label_name"]] = val_clip_counts.get(r["label_name"], 0) + 1

print("Train clip counts by class:", train_counts)
print("Val clip counts by class:", val_clip_counts)

x_train, y_train = train_ds[0]
x_val, y_val = val_ds[0]
print("Train sample shape:", tuple(x_train.shape), "label_id:", int(y_train))
print("Val sample shape:", tuple(x_val.shape), "label_id:", int(y_val))


Train clips: 29
Val 5s segments: 77
Train clip counts by class: {'background': 8, 'distress': 11, 'calm_speech': 10}
Val clip counts by class: {'background': 2, 'distress': 3, 'calm_speech': 3}
Train sample shape: (1, 37000) label_id: 0
Val sample shape: (1, 37000) label_id: 0


In [8]:
import random
from IPython.display import Audio, display

# Listen to one random sample from the training dataset
idx = random.randrange(len(train_ds))
x, y = train_ds[idx]  # x shape: [1, T]

id_to_label = {0: "background", 1: "calm_speech", 2: "distress"}
label_name = id_to_label.get(int(y), f"unknown({int(y)})")

audio_np = x.squeeze(0).detach().cpu().numpy()

print(f"Random dataset index: {idx}")
print(f"Label: {int(y)} ({label_name})")
print(f"Num samples: {audio_np.shape[0]}")

# Normalize for comfortable playback volume
peak = max(1e-8, float(abs(audio_np).max()))
audio_play = audio_np / peak

display(Audio(audio_play, rate=sample_rate))

Random dataset index: 11
Label: 1 (calm_speech)
Num samples: 37000
